In [1]:
# === Ô 1 (bản sửa): cài Playwright + trình duyệt cho đúng ===
!pip install -q playwright nest_asyncio
!python -m playwright install --with-deps chromium

# Kiểm tra trình duyệt đã có thật chưa
import glob
hits = glob.glob("/root/.cache/ms-playwright/**/chrome-headless-shell", recursive=True) \
     + glob.glob("/root/.cache/ms-playwright/**/headless_shell", recursive=True) \
     + glob.glob("/root/.cache/ms-playwright/**/chrome", recursive=True)
if hits:
    print("✅ Trình duyệt đã sẵn sàng:", hits[0])
    print("→ Chạy tiếp Ô 2.")
else:
    print("❌ Vẫn chưa thấy trình duyệt — gửi Claude dòng này.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 16.4 MB/s eta 0:00:00
Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,045 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRel

In [2]:
import nest_asyncio, asyncio
nest_asyncio.apply()
from playwright.async_api import async_playwright

TARGETS = [
    "https://finance.vietstock.vn/FPT/tai-tai-lieu.htm",     # Vietstock - trang tài liệu FPT
    "https://hnx.vn/vi-vn/thong-tin-cong-bo-ny-hnx.html",    # HNX - công bố thông tin
]
SKIP = ["google","facebook","gstatic","doubleclick","analytics","gtag",
        "cloudflare","jsdelivr","fontawesome","sentry","hotjar","zalo"]

async def probe(url):
    caught = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        page = await browser.new_page()
        async def on_resp(resp):
            u = resp.url
            if any(s in u.lower() for s in SKIP): return
            ct = resp.headers.get("content-type","")
            if "json" in ct or ("javascript" in ct and "api" in u.lower()):
                try: body = await resp.text()
                except: body = "(không đọc được)"
                caught.append((u, resp.status, body[:600]))
        page.on("response", on_resp)
        try:
            await page.goto(url, wait_until="networkidle", timeout=60000)
            await page.wait_for_timeout(6000)
        except Exception as e:
            print("  ⚠️", str(e)[:100])
        await browser.close()
    return caught

for url in TARGETS:
    print("="*90)
    print("🌐 TRANG:", url)
    print("="*90)
    res = asyncio.get_event_loop().run_until_complete(probe(url))
    print(f"→ Bắt được {len(res)} lệnh gọi dữ liệu:\n")
    for u, st, body in res:
        print("• API:", u)
        print("  Trạng thái:", st)
        print("  Mẫu dữ liệu:", body[:250].replace("\n"," "))
        print()

🌐 TRANG: https://finance.vietstock.vn/FPT/tai-tai-lieu.htm
  ⚠️ Page.goto: Timeout 60000ms exceeded.
Call log:
  - navigating to "https://finance.vietstock.vn/FPT/t
→ Bắt được 51 lệnh gọi dữ liệu:

• API: https://finance.vietstock.vn/Home/InitTheme
  Trạng thái: 200
  Mẫu dữ liệu: {"Init":200}

• API: https://gum.criteo.com/sid/json?origin=prebid&topUrl=https%3A%2F%2Ffinance.vietstock.vn%2F&domain=finance.vietstock.vn&cw=1&lsw=1
  Trạng thái: 200
  Mẫu dữ liệu: {"bundle":"XQdL7l9Jd2xsNnVjRnc3ZEUxSG4yUnY4czdnYXB2SyUyRjNjOFR2MG5rS1pFbHVOSmFLdUZpeXgzVWhSRzFxejF4V3I0T1dibDhhVGI1bmU1ZURVSzNSTHBKWTdqa0ZFalBwVTNIV0dneVVJR29JdlZBSHd5JTJCWGFRb0I5RjJqJTJGeVFoaXVVMg","bidId":"gNecLV9jcFBJQkdqWHBzTUw4Y2pHd3Q3bFQzQ3JJbD

• API: https://cdn.id5-sync.com/api/1.0/id5-api.js
  Trạng thái: 200
  Mẫu dữ liệu: /**  * @id5io/id5-api.js  * @version v1.0.97  * @link https://id5.io/  * @license Apache-2.0  */ !function(){"use strict";function r(t,e){var i,r=Object.keys(t);return Object.getOwnPropertySymbols&&